In [100]:
## Solution for question 1.an
## ref

import random

def create_augmented_matrix(A, b):
    return [row + [b_val] for row, b_val in zip(A, b)]

def print_augmented(M, n_cols_A, label=""):
    if label:
        print(label)
    for row in M:
        left  = "  ".join(f"{x:7.2f}" for x in row[:n_cols_A])
        right = "  ".join(f"{x:7.2f}" for x in row[n_cols_A:])
        print(f"  [ {left}  |  {right} ]")


def make_random_system(m, n, low=-9, high=9):
    """Random m×n matrix A and m×1 vector b"""
    assert m < n, "must must be smaller than n"
    A = [[random.randint(low, high) for _ in range(n)] for _ in range(m)]
    b = [random.randint(low, high) for _ in range(m)]
    return A, b

def ref(A, b):
    augmented_matrix = [row + [b_val] for row, b_val in zip(A, b)]
    rows,cols = len(A),len(A[0])
    pivot_row = 0
    pivot_cols = []
    for col in range(cols):
        if pivot_row >= rows:
            break

        pivot = None
        # find the pivot
        for r in range(pivot_row, rows):
            if abs(augmented_matrix[r][col]) > 1e-10:
                pivot = r
                break
        if pivot is None:
            continue

        # perform the swap
        augmented_matrix[pivot_row], augmented_matrix[pivot] = augmented_matrix[pivot], augmented_matrix[pivot_row]

        # perform elimination
        pivot_value = augmented_matrix[pivot_row][col]
        for r in range(pivot_row+1, rows):
            factor = augmented_matrix[r][col] / pivot_value
            for c in range(col, cols+1):
                augmented_matrix[r][c] = augmented_matrix[r][c] - factor * augmented_matrix[pivot_row][c]
        pivot_cols.append(col)
        pivot_row += 1

    return augmented_matrix, pivot_cols

A, b = make_random_system(4, 5)
ref_matrix, pivot_cols = ref(A, b)


print_augmented(ref_matrix,len(A[0]))
print("pivot columns:", pivot_cols)



  [    6.00     6.00     5.00     3.00     4.00  |    -1.00 ]
  [    0.00    10.00    -3.83    10.50     2.33  |     0.17 ]
  [    0.00     0.00    19.18    -6.85    -2.63  |    -0.62 ]
  [    0.00     0.00     0.00    -0.91     2.82  |    -3.00 ]
pivot columns: [0, 1, 2, 3]


In [101]:
## Solution for question 1.a
## rref
def rref(ref_matrix, pivot_cols):
    rows, cols = len(ref_matrix), len(ref_matrix[0])

    for i in range(len(pivot_cols) - 1, -1, -1):
        pr = i
        pc = pivot_cols[i]
        pivot_value = ref_matrix[pr][pc]

        # STEP A: scale the pivot row so the pivot becomes 1
        for c in range(cols):
            ref_matrix[pr][c] = ref_matrix[pr][c] / pivot_value

        # STEP B: clear entries ABOVE this pivot
        for r in range(pr):                                 # rows ABOVE only
            factor = ref_matrix[r][pc]                       # capture BEFORE modifying
            for c in range(cols):
                ref_matrix[r][c] = ref_matrix[r][c] - factor * ref_matrix[pr][c]   # subtract, and use [pr][c]

    return ref_matrix



A, b = make_random_system(4, 5)
ref_matrix, pivot_cols = ref(A, b)

print_augmented(ref_matrix,len(A[0]))
print("pivot columns:", pivot_cols)

rref_matrix = rref(ref_matrix, pivot_cols)
print("-------rref matrix-------")
print_augmented(rref_matrix,len(A[0]))


  [    6.00     0.00     2.00     6.00     9.00  |     1.00 ]
  [    0.00     1.00     3.00    -6.00     9.00  |     1.00 ]
  [    0.00     0.00     3.00   -29.00    22.00  |    10.00 ]
  [    0.00     0.00     0.00   208.00  -121.50  |   -77.50 ]
pivot columns: [0, 1, 2, 3]
-------rref matrix-------
  [    1.00     0.00     0.00     0.00     1.52  |     0.63 ]
  [    0.00     1.00     0.00     0.00     0.44  |    -0.43 ]
  [    0.00     0.00     1.00     0.00     1.69  |    -0.27 ]
  [    0.00     0.00     0.00     1.00    -0.58  |    -0.37 ]


In [109]:
## solution to question 1.b
##solving linear system
def find_solution(rref_matrix, pivot_cols, n):

    # 1. Non-pivot (free) columns — every A-column not in pivot_cols
    non_pivot_cols = [c for c in range(n) if c not in pivot_cols]

    # 2. Particular solution — set free vars to 0, read pivots from the b column
    particular = [0.0] * n
    for i, pc in enumerate(pivot_cols):
        particular[pc] = rref_matrix[i][n]      # b column is at index n

    # 3. Null space basis — one vector per free column
    null_basis = []
    for fc in non_pivot_cols:
        vec = [0.0] * n
        vec[fc] = 1.0                            # this free var = 1
        for i, pc in enumerate(pivot_cols):
            vec[pc] = -rref_matrix[i][fc]        # pivot vars = negated free-col entries
        null_basis.append(vec)

    return pivot_cols, non_pivot_cols, particular, null_basis


A, b = make_random_system(6, 9)
print("Original Matrix:")
print_augmented(create_augmented_matrix(A,b),len(A[0]))
ref_matrix, pivot_cols = ref(A, b)
rref_matrix = rref(ref_matrix, pivot_cols)
print("RREF:")
print_augmented(rref_matrix,len(A[0]))

pivot_cols, non_pivot_cols, particular, null_basis = find_solution(rref_matrix, pivot_cols, len(A[0]))

print("Pivot columns:    ", pivot_cols)
print("Non-pivot columns:", non_pivot_cols)
print("Particular xp:    ", [round(v, 3) for v in particular])
print(f"Null-space basis ({len(null_basis)} vectors — one per free column):")
for fc, vec in zip(non_pivot_cols, null_basis):
    print(f"  for x{fc}=1: {[round(v, 3) for v in vec]}")

Original Matrix:
  [    9.00    -1.00     6.00     6.00    -2.00    -1.00     0.00     4.00    -1.00  |     4.00 ]
  [    2.00    -5.00    -1.00     3.00     6.00    -9.00     7.00     7.00    -3.00  |     3.00 ]
  [    1.00    -7.00     2.00    -3.00     8.00     0.00     6.00     1.00    -1.00  |     0.00 ]
  [   -6.00     0.00    -5.00    -9.00    -6.00     9.00     1.00     3.00     9.00  |     9.00 ]
  [   -9.00    -9.00     9.00     0.00    -1.00     7.00     5.00     8.00     6.00  |     9.00 ]
  [   -3.00     2.00     1.00     6.00     1.00    -3.00     7.00    -9.00     8.00  |    -2.00 ]
RREF:
  [    1.00     0.00     0.00     0.00     0.00     0.00     0.14     0.09     0.04  |     0.32 ]
  [   -0.00     1.00    -0.00    -0.00    -0.00    -0.00    -2.06    -1.19    -1.75  |    -2.28 ]
  [    0.00     0.00     1.00     0.00     0.00     0.00    -3.40     1.37    -3.92  |    -1.96 ]
  [   -0.00    -0.00    -0.00     1.00    -0.00    -0.00     3.69    -2.11     4.35  |     1.64

In [122]:
## solution to question 1.c
#general solution x = xp + a.n
import numpy as np

def verify_solution(A,b,row_to_verify,particular, null_basis, alpha):
    print("General Solution: x = xp + ap")
    general_solution = np.array(particular) + np.array([x * alpha for x in  null_basis])
    print(f"General Solution:",general_solution)
    sum = 0.0;
    for x in range(len(A)):
        sum = sum + A[row_to_verify][x] * general_solution[x]

    print(f"calculated solution: {round(sum,2)} ~~ original b {b[row_to_verify]}")
    print("")

#test
verify_solution(A,b,0,particular, null_basis[0], 2)





General Solution: x = xp + ap
General Solution: [ 0.03409292  1.84734291  4.84208193 -5.7375963  -3.24995567 -4.4136815
  2.          0.          0.        ]
calculated solution: 4.0 ~~ original b 4

